# Credit Risk Validation & Backtest — Real Freddie Mac Data

Everything built so far in the credit-risk module (`dbt/models/credit_risk/`)
proves the *mechanics* work — the SQL is correct, the tests pass, the
numbers are internally consistent. That's not the same as evidence the
**methodology** (vintage curves, roll-rate transition matrices) actually
forecasts real credit performance. This notebook closes that gap: it runs
the same techniques against a real, public loan-performance dataset with a
genuine time-based holdout, and reports how accurate the forecasts
actually were.

**Data**: Freddie Mac's Single-Family Loan-Level Dataset, 2021 vintage
sample (50,000 loans, `data/external/freddie_mac/`, gitignored — see that
folder's AGENTS.md). This is **mortgage** data, not credit cards — the
loss *levels* won't match Synchrony's card portfolio. What's being
validated is whether the vintage/roll-rate *methodology* forecasts
accurately on a real monthly loan-performance panel — that's the part
that transfers.

Read this end to end before discussing it in an interview — like the rest
of this module, it's written to double as review material, not just a
script that produces numbers.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)

PROJECT_ROOT = Path("/Users/trustanprice/Desktop/Personal/ledgerone")
ORIG = PROJECT_ROOT / "data/external/freddie_mac/loan_originations.parquet"
PERF = PROJECT_ROOT / "data/external/freddie_mac/loan_performance.parquet"

# Time-based split: everything up to TRAIN_CUTOFF fits the transition
# matrix; everything after is held out for comparison. See section 2 for
# why this must be time-based, not random.
TRAIN_CUTOFF = "2024-09-01"
HORIZON = 18  # months of holdout compared
SEVERE = ("90-119 DPD", "120+/Charged-Off")

con = duckdb.connect()
print("Data loaded from:", ORIG.name, "and", PERF.name)

Data loaded from: loan_originations.parquet and loan_performance.parquet


## 1. Ingestion & bucket mapping (recap)

`src/ingest_freddie_mac_validation_data.py` parses the raw pipe-delimited
files and maps them to this project's grain — see that script's docstring
for the full column-layout verification (fetched Freddie Mac's current,
January 2026 General User Guide directly, then cross-checked every field
this notebook uses against its actual value distribution across all
50,000 loans, not just its documented position).

**The bucket mapping is a modeling decision worth stating plainly**:
Freddie Mac's 2-digit delinquency status code (`'00'`=current, `'01'`=30-59
days, `'02'`=60-89, `'03'`=90-119, higher = more months delinquent, `'RA'`=REO
acquisition) maps onto this project's existing 5-bucket scheme the same
way the synthetic module's buckets work. Critically: **mortgages don't
"charge off" the way credit cards do** — there's no single event that
zeroes the balance. `120+/Charged-Off` here is a **serious-delinquency /
foreclosure-adjacent proxy for loss**, not a literal charge-off. That's a
real, explicit difference from the synthetic card module this notebook is
validating against — say so out loud if asked.

In [2]:
originations = con.execute(f"select * from read_parquet('{ORIG}')").df()
performance = con.execute(f"select * from read_parquet('{PERF}')").df()

print(f"Accounts: {len(originations):,}")
print(f"Performance rows: {len(performance):,}")
print(f"Performance window: {performance.performance_month.min().date()} - {performance.performance_month.max().date()}")
print(f"Months on book: {performance.months_on_book.min()} - {performance.months_on_book.max()}")
print()
print("Origination cohorts (by first-payment-date quarter):")
print(originations.origination_quarter.value_counts().sort_index())

Accounts: 50,000
Performance rows: 2,537,054
Performance window: 2021-01-01 - 2026-03-01
Months on book: 0 - 62

Origination cohorts (by first-payment-date quarter):
origination_quarter
2021-01-01     3729
2021-04-01    12823
2021-07-01    12216
2021-10-01    13277
2022-01-01     7947
2022-04-01        1
2022-07-01        3
2022-10-01        3
2023-01-01        1
Name: count, dtype: int64


Five cohorts have meaningful sample size (2021-Q1 through 2022-Q1,
3.7K-13.3K loans each); a handful of loans with a first-payment-date
quarter beyond that (a few loans whose closing straddled the 2021/2022
boundary) have single-digit counts and are excluded from the headline
backtest metrics below as statistically meaningless — noted here rather
than silently dropped.

In [3]:
REAL_COHORTS = [pd.Timestamp(x) for x in
                ["2021-01-01", "2021-04-01", "2021-07-01", "2021-10-01", "2022-01-01"]]
print("Cohorts used for the backtest:", [c.strftime('%Y-Q') + str((c.month-1)//3+1) for c in REAL_COHORTS])

Cohorts used for the backtest: ['2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4', '2022-Q1']


## 2. Why a time-based split, not a random one

A **random** split (e.g. 80% of loan-months picked at random for training,
20% held out) would let the model see performance from *later* in a
loan's life while training on data that, chronologically, includes
information from *after* the holdout points it's being tested against.
That's leakage — the model would implicitly "know" how a loan turned out
before being asked to forecast it, which is exactly backwards from how
this model would actually be used in production (forecasting the future
from what's known *today*).

A **time-based** split — fit the matrix on data through a cutoff month,
forecast forward, compare against what actually happened after that
cutoff — is the only split that honestly answers "if I'd built this model
on what I knew as of the cutoff, how well would it have forecast what
actually happened next?" That's the real question a production model's
backtest has to answer.

In [4]:
cutoff_mob = con.execute(f'''
    select o.origination_quarter, date_diff('month', o.origination_quarter, date '{TRAIN_CUTOFF}') as cutoff_mob
    from read_parquet('{ORIG}') o
    where o.origination_quarter in ({",".join("'" + c.strftime('%Y-%m-%d') + "'" for c in REAL_COHORTS)})
    group by 1 order by 1
''').df()
print(f"Train cutoff: {TRAIN_CUTOFF}  |  Holdout horizon: {HORIZON} months")
print()
print("Months-on-book at cutoff, by cohort (this is where each cohort's holdout window starts):")
print(cutoff_mob)

Train cutoff: 2024-09-01  |  Holdout horizon: 18 months

Months-on-book at cutoff, by cohort (this is where each cohort's holdout window starts):
  origination_quarter  cutoff_mob
0          2021-01-01          44
1          2021-04-01          41
2          2021-07-01          38
3          2021-10-01          35
4          2022-01-01          32
